In [ ]:
from libraries import *
from parameters import *
from util import *

import pandas as pd
import patsy
import statsmodels.api as sm


In [ ]:
adata = sc.read_h5ad("./../Data/ComboScreen.h5ad")


In [ ]:
adata = adata[adata.obs[['ASCL1', 'KLF14', 'NEUROD1',
       'NEUROG1', 'NR3C1', 'NTC', 'SIM1', 'TET2', 'TWIST1', 'VSX1', 'ZNF385A',
       'ZNF547', 'ZNF660', 'ZNF776']].sum(axis=1) < 3]

In [ ]:
cols = ['ASCL1', 'KLF14', 'NEUROD1', 'NEUROG1', 'NR3C1', 'NTC', 'SIM1',
        'TET2', 'TWIST1', 'VSX1', 'ZNF385A', 'ZNF547', 'ZNF660', 'ZNF776']

def combine_onehot(row):
    # Select all column names where value == 1
    active = [col for col in cols if row[col] == 1]
    # Join multiple actives with '+', or return 'None' if none are active
    return '+'.join(active) if active else 'None'

adata.obs['perturbation'] = adata.obs[cols].apply(combine_onehot, axis=1)


In [ ]:
sc.pp.normalize_total(adata, target_sum=4000)
sc.pp.log1p(adata)


In [ ]:
adata

In [ ]:
adata=adata[~((adata.obs[['ASCL1', 'KLF14', 'NEUROD1',
       'NEUROG1', 'NR3C1', 'NTC', 'SIM1', 'TET2', 'TWIST1', 'VSX1', 'ZNF385A',
       'ZNF547', 'ZNF660', 'ZNF776']].sum(axis=1) == 2) & (adata.obs['NTC'] == 1)),]

In [ ]:

obs = adata.obs.copy()
# ensure time is coded (0/1) or categorical; here we treat it as categorical with baseline day04
obs["time_point"] = pd.Categorical(obs["time_point"], categories=["day04","day10"])

pert_cols = [c for c in obs.columns if c in ['ASCL1', 'KLF14', 'NEUROD1',
       'NEUROG1', 'NR3C1', 'SIM1', 'TET2', 'TWIST1', 'VSX1', 'ZNF385A',
       'ZNF547', 'ZNF660', 'ZNF776']]
P = " + ".join(pert_cols)

# Main perts + pairwise perts + time + pert×time + (pairwise)×time:
formula_rhs = f"C(time_point) + ({P}) + ({P}):C(time_point) + ({P}):({P}) + C(time_point):({P}):({P})"
# Full formula for one gene: y ~ <rhs>
# Build design once:
X = patsy.dmatrix(formula_rhs, obs, return_type="dataframe")



In [ ]:
X

In [ ]:
adata.var_names

In [ ]:
y = np.asarray(adata[:, "WASH7P"].X.A).ravel()

In [ ]:
y

In [ ]:
# Fit per gene (example for one gene)
fit = sm.OLS(y, X).fit()
print(fit.summary())
